# 03 — Coarse cell-type annotation

**Independently re-derive** the coarse cell-type annotation for one species from
marker expression, then compare against the collaborator's published labels as a
check on *both* analyses.

Run once per species: set `SPECIES` in the config cell. One parameterised
notebook rather than two files, so a fix in one species reaches the other. For a
rendered per-species record:

    papermill 03_annotate.ipynb runs/03_annotate_mouse.ipynb -p SPECIES mouse
    papermill 03_annotate.ipynb runs/03_annotate_human.ipynb -p SPECIES human

### Method

Follows Cevrim et al. Methods, *Data curation and cell type annotation*:
canonical markers evaluated per cluster; clusters with matching profiles merged
under one annotation; clusters with mixed profiles flagged and excluded.

Two deliberate deviations, both documented in the repo:

1. **Resolution.** The paper used clustree to pick 0.1. Cluster granularity at a
   *fixed* resolution is not preserved across independent scVI fits, so this
   notebook selects a resolution by marker-based criteria (cell 4). The
   reproducible claim is "a resolution exists that recovers the published cell
   types cleanly", not "0.1 does".
2. **Selection tool.** Marker-score sweep instead of clustree (an R package),
   keeping the stage in Python. Same job: choose the coarsest partition that
   represents every cell type.

### Discipline

The published marker panel and the list of cell types are in the paper — using
them is not peeking. The **answer key** is the per-cell `cell_type_annotations`
in the collaborator's deposited object, and it stays sealed until cell 7. Make
your calls in cell 6 before running cell 7.

### Nothing is recomputed

`02_integrate` already wrote all 14 Leiden resolutions into
`latent_{species}.h5ad`. This notebook chooses among them.

## 1. Config

In [ ]:
# Papermill parameter cell.
SPECIES = "mouse"          # "mouse" | "human"
REPO    = "/n/eddy_lab/users/nhilgert/xmens_final"

# Clusters whose top-two lineage scores differ by less than this are treated as
# ambiguous and held for manual adjudication (cells 6-8).
AMBIG_MARGIN = 0.05

# A cluster is called "decidual" rather than "fibroblast" when its decidual
# score clears this, even if the fibroblast score is nominally higher. See the
# nesting note in resources/markers/README.md.
NEST_THRESHOLD = 0.05

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import scanpy as sc
import pandas as pd, numpy as np
from pathlib import Path

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

REPO = Path(REPO)
LATENT = REPO / "data/interim" / f"latent_{SPECIES}.h5ad"
PANEL  = REPO / "resources/markers" / f"panel_{SPECIES}.tsv"
OUT    = REPO / "resources/markers" / f"annotations_{SPECIES}.tsv"

# The collaborator's deposited object -- the ANSWER KEY. Not loaded until cell 7.
SUBMISSION = Path("/n/eddy_lab/users/nhilgert/decidualization/revisions/adatas/submission")
TARGET = {"mouse": "20251008_Cevrim_XMens_All_Cells.h5ad",
          "human": "20251008_Cevrim_Human_All_Cells.h5ad"}[SPECIES]

# They renamed the mouse bleeding timepoint to "menstrual" to align with human
# nomenclature; our samples.tsv predates that. Needed for the barcode join.
SAMPLE_ALIAS = {"4dpibleedingmck11a": "4dpimenstrualmck11a"}

print(f"species : {SPECIES}")
print(f"latent  : {LATENT}")
print(f"panel   : {PANEL}")

## 2. Load the object and the marker panel

In [ ]:
adata = sc.read_h5ad(LATENT)
print(f"{adata.n_obs:,} cells x {adata.n_vars:,} genes")
print("obsm:", list(adata.obsm))

res_cols = sorted([c for c in adata.obs.columns if c.startswith("leiden_scVI_")],
                  key=lambda c: float(c.rsplit("_", 1)[1]))
print(f"{len(res_cols)} Leiden resolutions available:",
      [c.rsplit("_", 1)[1] for c in res_cols])

# integrate.py sets .X = log1p(normalize_total(counts, 1e4)); score_genes needs that.
xmax = float(adata.X.max())
print(f".X max = {xmax:.1f} ->", "log1p-normalised (good)" if xmax < 50
      else "LOOKS LIKE RAW COUNTS -- check integrate.py before scoring")

In [ ]:
panel = pd.read_csv(PANEL, sep="\t", comment="#")
panel["genes"] = panel["genes"].str.split(",")
panel["nested_within"] = panel["nested_within"].fillna("")
display(panel)

lineage_sets, context_sets, missing = {}, {}, {}
for _, r in panel.iterrows():
    have = [g for g in r.genes if g in adata.var_names]
    gone = [g for g in r.genes if g not in adata.var_names]
    if gone: missing[r.celltype] = gone
    (lineage_sets if r.role == "lineage" else context_sets)[r.celltype] = have

NESTED = {r.celltype: r.nested_within
          for _, r in panel.iterrows() if r.nested_within}
ORDER  = [r.celltype for _, r in panel.iterrows() if r.role == "lineage"]

if missing:
    print("MARKERS NOT IN var_names:", missing)
    print("  -> check gene symbol casing / reference build before continuing")
print("\nlineage sets:", {k: len(v) for k, v in lineage_sets.items()})
print("context sets:", {k: len(v) for k, v in context_sets.items()}, "(never used to assign)")
print("nesting     :", NESTED)

## 3. Score each lineage

`sc.tl.score_genes` subtracts a control set matched on expression bin, so sets of
different size are roughly comparable. Single-gene sets (mouse `Pdgfra`) are
still noisier — keep that in mind when reading small margins below.

In [ ]:
score_cols = []
for ct, genes in lineage_sets.items():
    if not genes:
        print(f"SKIP {ct}: no markers present"); continue
    col = f"score_{ct}"
    sc.tl.score_genes(adata, genes, score_name=col, ctrl_size=50, random_state=0)
    score_cols.append(col)

print(adata.obs[score_cols].describe().T[["mean", "std", "min", "max"]]
      .round(3).to_string())

## 4. Choose a resolution

Pick the **lowest** resolution where every expected cell type is the top-scoring
lineage of at least one cluster, and few clusters are ambiguous. Higher
resolutions always split further; we want the coarsest partition that still
represents every lineage.

This replaces the clustree step in the methods.

In [ ]:
def cluster_scores(res_col):
    cs = adata.obs.groupby(res_col, observed=True)[score_cols].mean()
    cs.columns = [c.replace("score_", "", 1) for c in cs.columns]
    return cs

rows = []
for col in res_cols:
    if float(col.rsplit("_", 1)[1]) > 1.0:      # coarse annotation only
        continue
    cs = cluster_scores(col)
    srt = np.sort(cs.values, axis=1)
    margin = srt[:, -1] - srt[:, -2]
    rows.append({
        "res": col.rsplit("_", 1)[1],
        "clusters": len(cs),
        "lineages_covered": cs.idxmax(axis=1).nunique(),
        "lineages_expected": len(score_cols),
        "median_margin": round(float(np.median(margin)), 3),
        "min_margin": round(float(margin.min()), 3),
        "n_ambiguous": int((margin < AMBIG_MARGIN).sum()),
    })
sweep = pd.DataFrame(rows).set_index("res")
sweep["complete"] = sweep.lineages_covered == sweep.lineages_expected
display(sweep)

ok = sweep[sweep.complete]
print("lowest complete resolution:", ok.index[0] if len(ok)
      else "NONE -- no resolution <=1.0 covers every lineage; extend the sweep "
           "or check the panel")

In [ ]:
# Set from the sweep above. Do not just take the suggestion -- look at cell 5 first.
RES = f"leiden_scVI_{sweep[sweep.complete].index[0]}" if (sweep.complete).any() \
      else res_cols[1]
print("using", RES, "->", adata.obs[RES].nunique(), "clusters")

## 5. Inspect: scores and markers per cluster

In [ ]:
cs = cluster_scores(RES)
print("mean lineage score per cluster:")
display(cs.round(3).style.background_gradient(axis=1, cmap="Reds"))

In [ ]:
plot_sets = {k: v for k, v in {**lineage_sets, **context_sets}.items() if v}
sc.pl.dotplot(adata, plot_sets, groupby=RES, standard_scale="var",
              title=f"{SPECIES}: markers by cluster ({RES})", show=True)

## 6. Make the calls — **before** unsealing the answer key

Argmax over lineage scores, then the nesting override: decidual cells are
differentiated fibroblasts and express both marker sets, so a cluster called
"fibroblast" whose decidual score is clearly positive is really decidual.

Anything flagged AMBIGUOUS goes back to the dotplot above and gets decided by
eye. A cluster with no dominant signature at all is the *mixed profile* case the
methods describe — a candidate for exclusion, not for a label.

In [ ]:
srt = np.sort(cs.values, axis=1)
calls = pd.DataFrame({
    "n":         adata.obs[RES].value_counts().reindex(cs.index).astype(int),
    "call":      cs.idxmax(axis=1),
    "runner_up": cs.apply(lambda r: r.nlargest(2).index[-1], axis=1),
    "top_score": srt[:, -1].round(3),
    "margin":    (srt[:, -1] - srt[:, -2]).round(3),
})

for child, parent in NESTED.items():
    if child not in cs.columns: continue
    flip = (calls["call"] == parent) & (cs[child] > NEST_THRESHOLD)
    if flip.any():
        print(f"nesting override {parent} -> {child}: clusters {list(cs.index[flip])}")
        calls.loc[flip, "call"] = child

calls["flag"] = np.where(calls.margin < AMBIG_MARGIN, "AMBIGUOUS", "")
calls["adjudication"] = ""          # fill in by hand after cell 8
display(calls)

print("lineages called:", sorted(calls.call.unique()))
print("expected       :", sorted(ORDER))
print("never called   :", sorted(set(ORDER) - set(calls.call)) or "none")
print("needs adjudication:", list(calls.index[calls.flag != ""]) or "none")

## 7. Unseal the answer key

Everything above was derived from markers alone. Now compare.

In [ ]:
def norm_sample(s):
    return s.astype(str).str.lower().str.replace(r"[^a-z0-9]", "", regex=True)

def join_key(obs):
    return (norm_sample(obs["sample"]).replace(SAMPLE_ALIAS) + "||" +
            obs["cell_barcode"].astype(str).str.replace(r"-\d+$", "", regex=True))

them = sc.read_h5ad(SUBMISSION / TARGET, backed="r").obs.copy()
them["k"] = join_key(them)

ref = join_key(adata.obs).map(them.set_index("k")["cell_type_annotations"])
adata.obs["celltype_ref"] = ref.astype(object).where(ref.notna(), "unlabeled").values

n_lab = int((adata.obs.celltype_ref != "unlabeled").sum())
print(f"theirs {len(them):,} | mine {adata.n_obs:,} | joined {n_lab:,} "
      f"({100*n_lab/len(them):.1f}% of theirs)")
print(f"unlabeled here = {adata.n_obs - n_lab:,} (cells they removed downstream)")
if n_lab < 0.9 * len(them):
    print("LOW JOIN RATE -- check sample naming / barcode suffixes before trusting cell 8")

In [ ]:
ct   = pd.crosstab(adata.obs[RES], adata.obs["celltype_ref"])
kept = ct[[c for c in ct.columns if c != "unlabeled"]]

cmp = pd.DataFrame({
    "n":          ct.sum(axis=1),
    "mine":       calls["call"],
    "theirs":     kept.idxmax(axis=1),
    "purity":     (kept.max(axis=1) / kept.sum(axis=1)).round(3),
    "pct_unlab":  (100 * ct.get("unlabeled", 0) / ct.sum(axis=1)).round(1),
    "margin":     calls["margin"],
})
cmp["agree"] = cmp.mine == cmp.theirs
display(cmp)

frac = cmp.loc[cmp.agree, "n"].sum() / cmp.n.sum()
print(f"agreement: {cmp.agree.sum()}/{len(cmp)} clusters, {100*frac:.1f}% of cells")
print(f"total unlabeled: {int(ct.get('unlabeled', 0).sum()):,}")

## 8. Adjudicate

For every disagreement there are exactly three outcomes, and each is worth
recording:

- **markers back me** — a correction to raise with the collaborators
- **markers back them** — my call was wrong; fix it in `calls`
- **genuinely ambiguous** — mixed profile; exclude the cluster and say so

Write the verdict into `calls["adjudication"]` before the freeze cell. This is
the part that checks the collaborators' work, so the reasoning belongs in the
lab notebook too, not only in the TSV.

In [ ]:
disagree = list(cmp.index[~cmp.agree]) + list(calls.index[calls.flag != ""])
disagree = sorted(set(disagree), key=lambda c: -cmp.loc[c, "n"])

for c in disagree:
    r = cmp.loc[c]
    print(f"\n=== cluster {c}   n={int(r.n):,}   mine={r.mine}   theirs={r.theirs}"
          f"   purity={r.purity}   unlabeled={r.pct_unlab}%   margin={r.margin}")
    print("   my scores  :", cs.loc[c].sort_values(ascending=False).head(3).round(3).to_dict())
    lab = ct.loc[c]; lab = lab[lab > 0].sort_values(ascending=False)
    print("   their labels:", lab.head(5).to_dict())

if disagree:
    sub = adata[adata.obs[RES].isin(disagree)].copy()
    sc.pl.dotplot(sub, plot_sets, groupby=RES, standard_scale="var",
                  title=f"{SPECIES}: disputed clusters", show=True)
else:
    print("no disagreements and nothing ambiguous")

In [ ]:
# --- manual overrides, after looking at the evidence above ---
# calls.loc["3", "call"]         = "vascular cells"
# calls.loc["3", "adjudication"] = "markers back me: Pecam1+ under a fibroblast label"
# calls.loc["5", "adjudication"] = "mixed profile, no dominant signature -> exclude"

EXCLUDE = []          # cluster IDs to drop as mixed-profile
calls["exclude"] = calls.index.isin([str(c) for c in EXCLUDE])
display(calls[["n", "call", "margin", "exclude", "adjudication"]])

## 9. Freeze

`annotations_{species}.tsv` is the input to `workflow/rules/04_annotate.smk`,
which is a pure lookup — no judgment at pipeline runtime.

`source_resolution` and `derived_from` are load-bearing: Leiden IDs are only
meaningful relative to one scVI fit, so this mapping is valid **for this
checkpoint only**. Re-running `02_integrate` invalidates it silently. See
`resources/markers/README.md`.

In [ ]:
final = calls[["call", "exclude", "margin", "n", "adjudication"]].copy()
final = final.rename(columns={"call": "celltype"})
final.index.name = "cluster"
final["source_resolution"] = RES
final["derived_from"] = LATENT.name

OUT.parent.mkdir(parents=True, exist_ok=True)
final.to_csv(OUT, sep="\t")
print(f"wrote {OUT}\n")
display(final)

TARGET_N = {"mouse": 57242, "human": 64553}[SPECIES]
kept_n = int(final.loc[~final.exclude, "n"].sum())
print(f"cells retained after cluster exclusion: {kept_n:,} / {adata.n_obs:,}")
print(f"published target: {TARGET_N:,}")
print(f"still to remove at lineage reintegration: {kept_n - TARGET_N:,}")

## Next

`04_lineage.ipynb`: split by the coarse annotation, reintegrate each lineage with
scVI, then apply the outlier rule from the methods — k=100 kNN graph **in the
lineage UMAP embedding** (not the scVI latent), drop cells whose mean neighbour
distance exceeds median + 3 MADs. That stage produces the fine subtype labels and
accounts for the remainder of the gap printed above.

The counts to close differ in character by species: mouse's gap is mostly
cluster-level exclusion (handled here), human's is mostly diffuse (handled
there).